In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics tqdm pillow -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
BASE = "/content/drive/MyDrive/CMPE 401/Instructor Defined 1"

!mkdir -p "{BASE}/VisDrone"

if not os.path.exists(f"{BASE}/VisDrone/VisDrone2019-DET-train/images"):
    !unzip -q -o "{BASE}/VisDrone2019-DET-train.zip"         -d "{BASE}/VisDrone/"
    !unzip -q -o "{BASE}/VisDrone2019-DET-val.zip"           -d "{BASE}/VisDrone/"
    !unzip -q -o "{BASE}/VisDrone2019-DET-test-challenge.zip" -d "{BASE}/VisDrone/"
    print("Done. Contents:")
    !ls "{BASE}/VisDrone/"
else:
    print("Already unzipped, skipping.")
    !ls "{BASE}/VisDrone/"

Already unzipped, skipping.
VisDrone2019-DET-test-challenge  VisDrone2019-DET-train  VisDrone2019-DET-val


In [ ]:
import os

base = "/content/drive/MyDrive/CMPE 401/Instructor Defined 1/VisDrone"

for root, dirs, files in os.walk(base):
    # Only show 2 levels deep
    level = root.replace(base, "").count(os.sep)
    if level < 2:
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level == 1:
            subindent = " " * 2 * (level + 1)
            print(f"{subindent}({len(files)} files)")

VisDrone/
  VisDrone2019-DET-val/
    (1 files)
  VisDrone2019-DET-test-challenge/
    (0 files)
  VisDrone2019-DET-train/
    (0 files)


In [ ]:
import os

BASE = "/content/drive/MyDrive/CMPE 401/Instructor Defined 1"
VISDRONE_DIR = f"{BASE}/VisDrone"

# Verify
for split in ["VisDrone2019-DET-train", "VisDrone2019-DET-val", "VisDrone2019-DET-test-challenge"]:
    split_dir = os.path.join(VISDRONE_DIR, split)
    print(f"\n=== {split} ===")
    for item in os.listdir(split_dir):
        subpath = os.path.join(split_dir, item)
        count = len(os.listdir(subpath)) if os.path.isdir(subpath) else ""
        print(f"  {item}/  ({count} files)" if os.path.isdir(subpath) else f"  {item}")


=== VisDrone2019-DET-train ===
  annotations/  (6471 files)
  images/  (6471 files)
  labels/  (6471 files)

=== VisDrone2019-DET-val ===
  .DS_Store
  annotations/  (548 files)
  images/  (548 files)
  labels/  (548 files)

=== VisDrone2019-DET-test-challenge ===
  images/  (1580 files)
  labels/  (0 files)


In [ ]:
import os
from PIL import Image
from tqdm import tqdm

SKIP_CLASSES = {0, 11}

def convert_split(images_dir, annots_dir, output_labels_dir):
    os.makedirs(output_labels_dir, exist_ok=True)
    annot_files = [f for f in os.listdir(annots_dir) if f.endswith(".txt")]
    skipped = 0

    for annot_file in tqdm(annot_files, desc=f"Converting {os.path.basename(images_dir)}"):
        name        = os.path.splitext(annot_file)[0]
        annot_path  = os.path.join(annots_dir, annot_file)
        image_path  = os.path.join(images_dir, name + ".jpg")
        output_path = os.path.join(output_labels_dir, annot_file)

        if not os.path.exists(image_path):
            skipped += 1
            continue

        img = Image.open(image_path)
        img_w, img_h = img.size

        yolo_lines = []
        with open(annot_path, "r") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue

                left   = int(parts[0])
                top    = int(parts[1])
                width  = int(parts[2])
                height = int(parts[3])
                cat_id = int(parts[5])

                if cat_id in SKIP_CLASSES:
                    continue

                # Clamp to image boundaries to avoid invalid boxes
                left   = max(0, left)
                top    = max(0, top)
                width  = min(width,  img_w - left)
                height = min(height, img_h - top)

                if width <= 0 or height <= 0:
                    continue

                x_center = (left + width  / 2) / img_w
                y_center = (top  + height / 2) / img_h
                norm_w   = width  / img_w
                norm_h   = height / img_h
                class_id = cat_id - 1

                yolo_lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
                )

        with open(output_path, "w") as f:
            f.write("\n".join(yolo_lines))

    print(f"Done. Skipped {skipped} files with no matching image.")

BASE = "/content/drive/MyDrive/CMPE 401/Instructor Defined 1/VisDrone"

# Convert train
convert_split(
    images_dir        = f"{BASE}/VisDrone2019-DET-train/images",
    annots_dir        = f"{BASE}/VisDrone2019-DET-train/annotations",
    output_labels_dir = f"{BASE}/VisDrone2019-DET-train/labels"
)

# Convert val
convert_split(
    images_dir        = f"{BASE}/VisDrone2019-DET-val/images",
    annots_dir        = f"{BASE}/VisDrone2019-DET-val/annotations",
    output_labels_dir = f"{BASE}/VisDrone2019-DET-val/labels"
)

Converting images: 100%|██████████| 6471/6471 [03:25<00:00, 31.42it/s]


Done. Skipped 0 files with no matching image.


Converting images: 100%|██████████| 548/548 [00:13<00:00, 41.57it/s]

Done. Skipped 0 files with no matching image.


In [ ]:
BASE = "/content/drive/MyDrive/CMPE 401/Instructor Defined 1"
VISDRONE_DIR = f"{BASE}/VisDrone"

print("Zipping...")
os.system(f'zip -r "/content/VisDrone_YOLO.zip" "{VISDRONE_DIR}"')
print("Done. Size:")
os.system('du -sh /content/VisDrone_YOLO.zip')

Zipping...
Done. Size:


0

In [ ]:
from google.colab import files
files.download('/content/VisDrone_YOLO.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>